In [0]:
from pyspark.sql.functions import col, to_timestamp, expr

df = spark.read.table("automobile_catalog.001_bronze.`order`")

# Remove duplicates
df = df.dropDuplicates()

# Timestamp format
date_format = "dd-MM-yyyy HH:mm"

cols = [
    "vehicle_in_datetime",
    "vehicle_out_datetime",
    "planned_work_start_datetime",
    "actual_work_start_datetime",
    "planned_completion_datetime",
    "actual_completion_datetime",
    "promised_delivery_datetime",
    "actual_delivery_datetime"
]

for c in cols:
    df = df.withColumn(c, expr(f"try_to_timestamp({c}, '{date_format}')"))

# Filter valid records
df = df.filter(col("order_id").isNotNull())

# Write to silver
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("automobile_catalog.002_silver.order")